### **Extraer el `precio`, `descripción` y `ubicación` de los anuncios en la pagina de OLX**

Vamos a scrapear el siguiente sitio:

https://www.olx.in/cars_c84

`ATENCION`: OLX necesita que le demos permisos de geolocalizacion al navegador de selenium para que nos muestre los datos. Esto lo haremos una unica vez en la primer corrida del programa. Este problema es mas comun en usuarios de MAC

Vamos a inspeccionar el sitioweb:

<center><img src="https://i.postimg.cc/ZKD0V0Dw/ws-313.png"></center>

Vamos a buscar el boton de **`Load more`** para pulsarlo 3 veces y cargar más articulos:

<center><img src="https://i.postimg.cc/7ZGhcDFw/ws-314.png"></center>

Buscamos la caja que almacena la información de cada articulo:

<center><img src="https://i.postimg.cc/vm5BYh8K/ws-315.png"></center>

Buscamos su **`precio`**:

<center><img src="https://i.postimg.cc/B6svXFXn/ws-316.png"></center>

Buscamos su **`descripción`**:

<center><img src="https://i.postimg.cc/1X35KZ0x/ws-317.png"></center>

Buscamos su **`ubicación`**:

<center><img src="https://i.postimg.cc/G2QmYpyJ/ws-318.png"></center>

Por tanto, generamos el siguiente código:

In [5]:
import random
from time import sleep
from selenium.webdriver.common.by import By
from selenium import webdriver
#from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
#from webdriver_manager.chrome import ChromeDriverManager
import pandas as pd

# Asi podemos setear el user-agent en selenium
options = Options()
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36")

website = "https://www.olx.in"
# Se recomienda utilizar Chrome, pero podriamos utilizar Firefox, Safari, Edge, etc.
# driver = webdriver.Chrome(service = Service(ChromeDriverManager().install()), options=options)
driver = webdriver.Chrome(options=options)
driver.get(website)
driver.maximize_window()

# driver.refresh() # Solucion de un bug extraño en Windows en donde los anuncios solo cargan al hacerle refresh a la página
# sleep(2) # Esperamos que cargue el boton

# Busco el boton para cargar mas informacion
boton = driver.find_element(By.XPATH, '//button[@data-aut-id="btnLoadMore"]')
for i in range(3): # Voy a darle click en cargar mas 3 veces
    try:
        # Le doy click al boton
        boton.click()
        # Espero que cargue la informacion dinamica
        # Devuelve un número aleatorio entre, e incluido, 8 y 10:
        sleep(random.uniform(8.0, 10.0))
        # Buscar el boton nuevamente para darle click en la siguiente iteracion
        boton = driver.find_element(By.XPATH, '//button[@data-aut-id="btnLoadMore"]')
    except:
        # Si hay algun error, rompo el lazo.
        break

# Encuentro cual es el XPATH de cada elemento donde esta la informacion que quiero extraer
# Esto es una LISTA. Por eso el metodo esta en plural
articulos = driver.find_elements(By.XPATH, '//li[@data-aut-id="itemBox"]')

# Inicializando el almacenamiento
articulo_precio = []
articulo_descripcion = []
articulo_ubicacion = []

# Recorro cada uno de los anuncios que he encontrado
for articulo in articulos:
    try:
        # Por cada anuncio extraigo el precio
        articulo_precio.append(articulo.find_element(By.XPATH, './/span[@data-aut-id="itemPrice"]').text)
        # Por cada anuncio extraigo la descripcion
        articulo_descripcion.append(articulo.find_element(By.XPATH, './/span[@data-aut-id="itemTitle"]').text)
        # Por cada anuncio extraigo la ubicacion
        articulo_ubicacion.append(articulo.find_element(By.XPATH, './/span[@data-aut-id="item-location"]').text)
    except Exception as e:
        f'Anuncio carece de precio o descripcion'

driver.quit()

# Almacenar los datos en un DataFrame y exportar a un archivo CSV
df_articulos = pd.DataFrame({'precio': articulo_precio, 'descripcion': articulo_descripcion, 'ubicacion': articulo_ubicacion})
df_articulos.to_csv('articulos.csv', index=False)